Evaluating Instruction Responses Using the OpenAI API

In [ ]:
from importlib.metadata import version

pkgs = ["openai",  # OpenAI API
        "tqdm",    # Progress bar
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
import json
from openai import OpenAI

# 从 config.json 读取 OpenAI API key（换成你自己的 key；用文件避免把密钥写进 notebook）
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)

In [ ]:
# 封装一次 GPT-4 调用作为「裁判」；temperature=0 + seed=123 让评分尽量确定、可复现
def run_chatgpt(prompt, client, model="gpt-4-turbo"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        seed=123,
    )
    return response.choices[0].message.content


prompt = "Respond with 'hello world' if you got this message."  # 连通性测试
run_chatgpt(prompt, client)

In [ ]:
json_file = "eval-example-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

In [ ]:
def format_input(entry):
    # 按 Alpaca 风格把一条样本拼成给「裁判模型」看的指令文本
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""  # 有 input 才拼这段
    # 【bug修复】原此处有一行独立的 `instruction_text + input_text`：它只是计算后把结果丢弃(死代码)，
    # 没有赋值也没有返回，纯属冗余。已删除；真正的返回在下一行。
    return instruction_text + input_text  # 指令段 + 可选输入段

In [ ]:
# 前 5 条试跑：让 GPT-4 给 model 1 的回复打分，检查评分是否合理
for entry in json_data[:5]:
    prompt = (f"Given the input `{format_input(entry)}` "
              f"and correct output `{entry['output']}`, "
              f"score the model response `{entry['model 1 response']}`"
              f" on a scale from 0 to 100, where 100 is the best score. "
              )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model 1 response"])
    print("\nScore:")
    print(">>", run_chatgpt(prompt, client))
    print("\n-------------------------")

In [ ]:
from tqdm import tqdm


# 批量用 GPT-4 为某模型回复打分
def generate_model_scores(json_data, json_key, client):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the number only."  # 要求只回数字，便于 int() 解析
        )
        score = run_chatgpt(prompt, client)
        try:
            scores.append(int(score))
        except ValueError:
            continue  # 解析失败(没只回数字)就跳过该条

    return scores

In [ ]:
from pathlib import Path

# 分别为 model 1 / model 2 打分，打印平均分并保存到 scores/ 目录
for model in ("model 1 response", "model 2 response"):

    scores = generate_model_scores(json_data, model, client)
    print(f"\n{model}")
    print(f"Number of scores: {len(scores)} of {len(json_data)}")
    print(f"Average score: {sum(scores)/len(scores):.2f}\n")

    # Optionally save the scores（注意 scores/ 目录需已存在，否则 open(...,'w') 会报错）
    save_path = Path("scores") / f"gpt4-{model.replace(' ', '-')}.json"
    with open(save_path, "w") as file:
        json.dump(scores, file)